In [22]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents

# 1. Initialize the homework embedder
embedder = Embedder()

# 2. Fetch the 72 pinned markdown documents from GitHub
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]


In [24]:
# 1. Define and encode the query
q1_text = "How does approximate nearest neighbor search work?"
v = embedder.encode(q1_text)

# 2. Print out the first value
print(f"Q1 Answer (v[0]): {v[0]:.2f}")


Q1 Answer (v[0]): -0.02


In [25]:
# 1. Locate the exact target document by filename
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_file)

# 2. Embed the page content
v_doc = embedder.encode(target_doc["content"])

# 3. Calculate dot product (cosine similarity because vectors are pre-normalized)
similarity = np.dot(v, v_doc)

print(f"Q2 Answer (Cosine Similarity): {similarity:.2f}")


Q2 Answer (Cosine Similarity): 0.36


In [26]:
# 1. Chunk documents into overlapping components
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Create the matrix X using batch processing
chunk_texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_texts)

# 3. Compute vector-matrix dot product scores
scores = X.dot(v)

# 4. Locate the highest scoring chunk index
best_chunk_idx = np.argmax(scores)
best_chunk = chunks[best_chunk_idx]

print(f"Q3 Answer (Highest scoring file): {best_chunk['filename']}")
print(f"Top Matrix Score: {scores[best_chunk_idx]:.4f}")


Q3 Answer (Highest scoring file): 02-vector-search/lessons/07-sqlitesearch-vector.md
Top Matrix Score: 0.6489


In [29]:
import numpy as np

# 1. Define the Q4 evaluation target query
q4_text = "What metric do we use to evaluate a search engine?"

# 2. Encode and flatten into a clean 1D array
v_q4 = embedder.encode(q4_text).flatten()

# 3. VECTOR SEARCH BY HAND (This is exactly what minsearch.VectorSearch does under the hood)
# Computes the cosine similarities across all chunks at once
q4_scores = X.dot(v_q4)

# 4. Use the course sorting trick to find the highest-scoring chunk's index
best_chunk_idx = np.argsort(-q4_scores)[0]

# 5. Extract the source filename to answer your homework question
final_q4_doc = chunks[best_chunk_idx]

print("==================================================")
print(f"🎯 Q4 Homework Answer: {final_q4_doc['filename']}")
print(f"📈 Top Match Matrix Score: {q4_scores[best_chunk_idx]:.4f}")
print("==================================================")


🎯 Q4 Homework Answer: 04-evaluation/lessons/05-search-metrics.md
📈 Top Match Matrix Score: 0.5986


In [32]:
import numpy as np

# 1. Define your Q5 evaluation target query
q5_text = "How do I store vectors in PostgreSQL?"

# 2. Vector Search (By Hand Matrix Math to bypass the library bug)
v_q5 = embedder.encode(q5_text).flatten()
q5_vector_scores = X.dot(v_q5)

# Extract top 5 chunks according to vector similarity
top5_vector_indices = np.argsort(-q5_vector_scores)[:5]
vector_results = [chunks[idx] for idx in top5_vector_indices]

# 3. Text Search (Keyword engine works perfectly)
text_results = t_index.search(query=q5_text, num_results=5)

# 4. Compare the files in each resulting pool
text_files = {doc["filename"] for doc in text_results}
vector_files = {doc["filename"] for doc in vector_results}

# Identify the file found ONLY by vector search
unique_to_vector = vector_files - text_files

print("==================================================")
print(f"🎯 Q5 Homework Answer (Unique to Vector):")
print(unique_to_vector)
print("==================================================")


🎯 Q5 Homework Answer (Unique to Vector):
{'02-vector-search/lessons/08-pgvector.md'}


In [33]:
# 1. Re-declare the course RRF scoring function
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            # Create a tuple key using the metadata structure of the chunk
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 2. Define the target query for the tools interface
q6_text = "How do I give the model access to tools?"

# 3. Generate Vector Results via manual matrix math
v_q6 = embedder.encode(q6_text).flatten()
q6_vector_scores = X.dot(v_q6)
top5_v6_indices = np.argsort(-q6_vector_scores)[:5]
q6_vector_results = [chunks[idx] for idx in top5_v6_indices]

# 4. Generate Text Results via keyword index search
q6_text_results = t_index.search(query=q6_text, num_results=5)

# 5. Fuse both lists together using Reciprocal Rank Fusion
fused_results = rrf([q6_vector_results, q6_text_results])

print("==================================================")
print(f"🎯 Q6 Homework Answer (Top Fused Hybrid File):")
print(f"👉 {fused_results[0]['filename']}")
print("==================================================")


🎯 Q6 Homework Answer (Top Fused Hybrid File):
👉 01-agentic-rag/lessons/13-function-calling.md
